# OULAD Exploration and Training

This notebook implements Milestone 1 of the AI Academic Mentor project:
- Load and explore OULAD dataset
- Feature engineering and data transformation
- Train a GradientBoostingClassifier for student outcome prediction
- Save models and encoders for deployment


In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import joblib
import os
from pathlib import Path

# Set up paths
data_path = Path("../Data_csv's")
models_path = Path("../6_Models")

# Create models directory if it doesn't exist
models_path.mkdir(exist_ok=True)

print("Libraries imported successfully!")
print(f"Data path: {data_path}")
print(f"Models path: {models_path}")


Libraries imported successfully!
Data path: ..\Data_csv's
Models path: ..\6_Models


## 1. Data Loading

Load the key CSV files: studentInfo.csv, studentAssessment.csv, and assessments.csv


In [2]:
# Load the key CSV files
print("Loading OULAD datasets...")

# Load studentInfo.csv
student_info = pd.read_csv(data_path / "studentInfo.csv")
print(f"studentInfo shape: {student_info.shape}")
print(f"Columns: {list(student_info.columns)}")

# Load studentAssessment.csv
student_assessment = pd.read_csv(data_path / "studentAssessment.csv")
print(f"\nstudentAssessment shape: {student_assessment.shape}")
print(f"Columns: {list(student_assessment.columns)}")

# Load assessments.csv
assessments = pd.read_csv(data_path / "assessments.csv")
print(f"\nassessments shape: {assessments.shape}")
print(f"Columns: {list(assessments.columns)}")

# Display basic info about each dataset
print("\n=== studentInfo Info ===")
print(student_info.info())
print("\n=== studentAssessment Info ===")
print(student_assessment.info())
print("\n=== assessments Info ===")
print(assessments.info())


Loading OULAD datasets...
studentInfo shape: (32593, 12)
Columns: ['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result']

studentAssessment shape: (173912, 5)
Columns: ['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score']

assessments shape: (206, 6)
Columns: ['code_module', 'code_presentation', 'id_assessment', 'assessment_type', 'date', 'weight']

=== studentInfo Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   code_module           32593 non-null  object
 1   code_presentation     32593 non-null  object
 2   id_student            32593 non-null  int64 
 3   gender                32593 non-null  object
 4   region                32593 non-null  object
 5   highest

## 2. Data Exploration

Let's explore the data to understand the structure and identify key patterns


In [3]:
# Explore studentInfo dataset
print("=== studentInfo Exploration ===")
print("\nFirst few rows:")
print(student_info.head())

print("\nFinal result distribution:")
print(student_info['final_result'].value_counts())

print("\nMissing values:")
print(student_info.isnull().sum())

print("\nData types:")
print(student_info.dtypes)


=== studentInfo Exploration ===

First few rows:
  code_module code_presentation  id_student gender                region  \
0         AAA             2013J       11391      M   East Anglian Region   
1         AAA             2013J       28400      F              Scotland   
2         AAA             2013J       30268      F  North Western Region   
3         AAA             2013J       31604      F     South East Region   
4         AAA             2013J       32885      F  West Midlands Region   

       highest_education imd_band age_band  num_of_prev_attempts  \
0       HE Qualification  90-100%     55<=                     0   
1       HE Qualification   20-30%    35-55                     0   
2  A Level or Equivalent   30-40%    35-55                     0   
3  A Level or Equivalent   50-60%    35-55                     0   
4     Lower Than A Level   50-60%     0-35                     0   

   studied_credits disability final_result  
0              240          N         Pa

In [4]:
# Explore studentAssessment dataset
print("=== studentAssessment Exploration ===")
print("\nFirst few rows:")
print(student_assessment.head())

print("\nScore statistics:")
print(student_assessment['score'].describe())

print("\nMissing values:")
print(student_assessment.isnull().sum())

print("\nUnique students with assessments:")
print(f"Number of unique students: {student_assessment['id_student'].nunique()}")

print("\nAssessment distribution by student:")
assessment_counts = student_assessment['id_student'].value_counts()
print(f"Min assessments per student: {assessment_counts.min()}")
print(f"Max assessments per student: {assessment_counts.max()}")
print(f"Mean assessments per student: {assessment_counts.mean():.2f}")


=== studentAssessment Exploration ===

First few rows:
   id_assessment  id_student  date_submitted  is_banked  score
0           1752       11391              18          0   78.0
1           1752       28400              22          0   70.0
2           1752       31604              17          0   72.0
3           1752       32885              26          0   69.0
4           1752       38053              19          0   79.0

Score statistics:
count    173739.000000
mean         75.799573
std          18.798107
min           0.000000
25%          65.000000
50%          80.000000
75%          90.000000
max         100.000000
Name: score, dtype: float64

Missing values:
id_assessment       0
id_student          0
date_submitted      0
is_banked           0
score             173
dtype: int64

Unique students with assessments:
Number of unique students: 23369

Assessment distribution by student:
Min assessments per student: 1
Max assessments per student: 28
Mean assessments per student

## 3. Feature Engineering

Calculate aggregated features from studentAssessment.csv and merge with studentInfo.csv


In [5]:
# Feature Engineering: Calculate aggregated features from studentAssessment
print("Calculating aggregated features from studentAssessment...")

# Calculate average score and number of assessments per student
assessment_aggregated = student_assessment.groupby('id_student').agg({
    'score': ['mean', 'count'],
    'date_submitted': ['min', 'max']  # Additional features for future use
}).reset_index()

# Flatten column names
assessment_aggregated.columns = ['id_student', 'avg_score', 'num_assessments', 'first_submission', 'last_submission']

# Handle missing scores by filling with 0 (students who didn't submit)
assessment_aggregated['avg_score'] = assessment_aggregated['avg_score'].fillna(0)
assessment_aggregated['num_assessments'] = assessment_aggregated['num_assessments'].fillna(0)

print(f"Aggregated assessment features shape: {assessment_aggregated.shape}")
print("\nFirst few rows of aggregated features:")
print(assessment_aggregated.head())

print("\nAggregated features statistics:")
print(assessment_aggregated[['avg_score', 'num_assessments']].describe())


Calculating aggregated features from studentAssessment...
Aggregated assessment features shape: (23369, 5)

First few rows of aggregated features:
   id_student  avg_score  num_assessments  first_submission  last_submission
0        6516  61.800000                5                17              210
1        8462  87.000000                7                -1               85
2       11391  82.000000                5                18              212
3       23629  82.500000                4                 9               95
4       23698  74.444444                9                21              243

Aggregated features statistics:
          avg_score  num_assessments
count  23369.000000     23369.000000
mean      73.086265         7.434593
std       15.667052         4.225845
min        0.000000         0.000000
25%       65.000000         4.000000
50%       76.000000         7.000000
75%       84.333333        11.000000
max      100.000000        28.000000


In [6]:
# Merge aggregated features with studentInfo
print("Merging aggregated features with studentInfo...")

# Merge the aggregated assessment features with student info
merged_data = student_info.merge(assessment_aggregated, on='id_student', how='left')

# Fill missing values for students who have no assessments
merged_data['avg_score'] = merged_data['avg_score'].fillna(0)
merged_data['num_assessments'] = merged_data['num_assessments'].fillna(0)

print(f"Merged data shape: {merged_data.shape}")
print(f"Original studentInfo shape: {student_info.shape}")
print(f"Students with assessments: {merged_data['num_assessments'].gt(0).sum()}")
print(f"Students without assessments: {merged_data['num_assessments'].eq(0).sum()}")

print("\nFirst few rows of merged data:")
print(merged_data.head())


Merging aggregated features with studentInfo...
Merged data shape: (32593, 16)
Original studentInfo shape: (32593, 12)
Students with assessments: 26727
Students without assessments: 5866

First few rows of merged data:
  code_module code_presentation  id_student gender                region  \
0         AAA             2013J       11391      M   East Anglian Region   
1         AAA             2013J       28400      F              Scotland   
2         AAA             2013J       30268      F  North Western Region   
3         AAA             2013J       31604      F     South East Region   
4         AAA             2013J       32885      F  West Midlands Region   

       highest_education imd_band age_band  num_of_prev_attempts  \
0       HE Qualification  90-100%     55<=                     0   
1       HE Qualification   20-30%    35-55                     0   
2  A Level or Equivalent   30-40%    35-55                     0   
3  A Level or Equivalent   50-60%    35-55          

## 4. Data Cleaning & Transformation

Prepare the data for modeling by handling missing values and encoding categorical features


In [7]:
# Data Cleaning
print("=== Data Cleaning ===")

# Check for missing values
print("Missing values in merged data:")
print(merged_data.isnull().sum())

# Handle missing values
print("\nHandling missing values...")

# Fill missing values for categorical features
categorical_columns = ['gender', 'region', 'highest_education', 'imd_band', 'age_band', 'disability']
for col in categorical_columns:
    if col in merged_data.columns:
        merged_data[col] = merged_data[col].fillna('Unknown')
        print(f"Filled missing values in {col} with 'Unknown'")

# Fill missing values for numerical features
numerical_columns = ['num_of_prev_attempts', 'studied_credits']
for col in numerical_columns:
    if col in merged_data.columns:
        merged_data[col] = merged_data[col].fillna(0)
        print(f"Filled missing values in {col} with 0")

print("\nMissing values after cleaning:")
print(merged_data.isnull().sum())


=== Data Cleaning ===
Missing values in merged data:
code_module                0
code_presentation          0
id_student                 0
gender                     0
region                     0
highest_education          0
imd_band                1111
age_band                   0
num_of_prev_attempts       0
studied_credits            0
disability                 0
final_result               0
avg_score                  0
num_assessments            0
first_submission        5847
last_submission         5847
dtype: int64

Handling missing values...
Filled missing values in gender with 'Unknown'
Filled missing values in region with 'Unknown'
Filled missing values in highest_education with 'Unknown'
Filled missing values in imd_band with 'Unknown'
Filled missing values in age_band with 'Unknown'
Filled missing values in disability with 'Unknown'
Filled missing values in num_of_prev_attempts with 0
Filled missing values in studied_credits with 0

Missing values after cleaning:
code_mod

In [8]:
# Feature Selection and Encoding
print("=== Feature Selection and Encoding ===")

# Select features for modeling
feature_columns = ['gender', 'disability', 'num_of_prev_attempts', 'studied_credits', 'avg_score', 'num_assessments']
target_column = 'final_result'

print(f"Selected features: {feature_columns}")
print(f"Target variable: {target_column}")

# Create feature matrix and target vector
X = merged_data[feature_columns].copy()
y = merged_data[target_column].copy()

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

# Check feature distributions
print("\nFeature distributions:")
for col in feature_columns:
    print(f"\n{col}:")
    print(X[col].value_counts().head())


=== Feature Selection and Encoding ===
Selected features: ['gender', 'disability', 'num_of_prev_attempts', 'studied_credits', 'avg_score', 'num_assessments']
Target variable: final_result

Feature matrix shape: (32593, 6)
Target vector shape: (32593,)

Feature distributions:

gender:
gender
M    17875
F    14718
Name: count, dtype: int64

disability:
disability
N    29429
Y     3164
Name: count, dtype: int64

num_of_prev_attempts:
num_of_prev_attempts
0    28421
1     3299
2      675
3      142
4       39
Name: count, dtype: int64

studied_credits:
studied_credits
60     16751
120     6328
30      3749
90      3144
180      830
Name: count, dtype: int64

avg_score:
avg_score
0.0     5898
80.0     220
70.0     192
75.0     191
76.0     176
Name: count, dtype: int64

num_assessments:
num_assessments
0.0     5866
12.0    3635
5.0     2880
7.0     2454
11.0    2348
Name: count, dtype: int64


In [9]:
# Label Encoding for categorical features and target
print("=== Label Encoding ===")

# Create and fit encoders for categorical features
categorical_features = ['gender', 'disability']
feature_encoders = {}

for feature in categorical_features:
    if feature in X.columns:
        encoder = LabelEncoder()
        X[feature] = encoder.fit_transform(X[feature].astype(str))
        feature_encoders[feature] = encoder
        print(f"Encoded {feature}: {len(encoder.classes_)} unique values")

# Create and fit encoder for target variable
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y.astype(str))

print(f"\nEncoded target variable: {len(target_encoder.classes_)} unique values")
print(f"Target classes: {target_encoder.classes_}")
print(f"Target distribution: {np.bincount(y_encoded)}")

# Display encoded features
print(f"\nEncoded feature matrix shape: {X.shape}")
print("First few rows of encoded features:")
print(X.head())


=== Label Encoding ===
Encoded gender: 2 unique values
Encoded disability: 2 unique values

Encoded target variable: 4 unique values
Target classes: ['Distinction' 'Fail' 'Pass' 'Withdrawn']
Target distribution: [ 3024  7052 12361 10156]

Encoded feature matrix shape: (32593, 6)
First few rows of encoded features:
   gender  disability  num_of_prev_attempts  studied_credits  avg_score  \
0       1           0                     0              240       82.0   
1       0           0                     0               60       66.4   
2       0           1                     0               60        0.0   
3       0           0                     0               60       76.0   
4       0           0                     0               60       54.4   

   num_assessments  
0              5.0  
1              5.0  
2              0.0  
3              5.0  
4              5.0  


## 5. Model Training

Train a GradientBoostingClassifier and evaluate its performance


In [10]:
# Split data into train and test sets
print("=== Train-Test Split ===")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Training target distribution: {np.bincount(y_train)}")
print(f"Test target distribution: {np.bincount(y_test)}")


=== Train-Test Split ===
Training set shape: (26074, 6)
Test set shape: (6519, 6)
Training target distribution: [2419 5641 9889 8125]
Test target distribution: [ 605 1411 2472 2031]


In [11]:
# Train GradientBoostingClassifier
print("=== Training GradientBoostingClassifier ===")

# Initialize and train the model
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

print("Training model...")
gb_model.fit(X_train, y_train)

print("Model training completed!")
print(f"Model parameters: {gb_model.get_params()}")

# Make predictions
y_pred = gb_model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {accuracy:.4f}")

# Display classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_encoder.classes_))


=== Training GradientBoostingClassifier ===
Training model...
Model training completed!
Model parameters: {'ccp_alpha': 0.0, 'criterion': 'friedman_mse', 'init': None, 'learning_rate': 0.1, 'loss': 'log_loss', 'max_depth': 6, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 100, 'n_iter_no_change': None, 'random_state': 42, 'subsample': 1.0, 'tol': 0.0001, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}

Model Accuracy: 0.6480

Classification Report:
              precision    recall  f1-score   support

 Distinction       0.62      0.46      0.53       605
        Fail       0.49      0.29      0.36      1411
        Pass       0.67      0.87      0.75      2472
   Withdrawn       0.69      0.69      0.69      2031

    accuracy                           0.65      6519
   macro avg       0.62      0.57      0.58      6519
weighted avg       0.63    

In [12]:
# Display confusion matrix
print("=== Confusion Matrix ===")
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# Create a more readable confusion matrix
cm_df = pd.DataFrame(cm, 
                     index=target_encoder.classes_, 
                     columns=target_encoder.classes_)
print("\nConfusion Matrix (with labels):")
print(cm_df)

# Feature importance
print("\n=== Feature Importance ===")
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': gb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature Importance:")
print(feature_importance)


=== Confusion Matrix ===
Confusion Matrix:
[[ 276    3  315   11]
 [   6  406  457  542]
 [ 142  107 2145   78]
 [  18  319  297 1397]]

Confusion Matrix (with labels):
             Distinction  Fail  Pass  Withdrawn
Distinction          276     3   315         11
Fail                   6   406   457        542
Pass                 142   107  2145         78
Withdrawn             18   319   297       1397

=== Feature Importance ===
Feature Importance:
                feature  importance
5       num_assessments    0.614687
4             avg_score    0.277584
3       studied_credits    0.061631
2  num_of_prev_attempts    0.023316
0                gender    0.015206
1            disability    0.007576


## 6. Save Models and Encoders

Save the trained model and encoders for deployment


In [13]:
# Save the trained model
print("=== Saving Models and Encoders ===")

# Save the trained GradientBoostingClassifier
model_path = models_path / "trained_model.pkl"
joblib.dump(gb_model, model_path)
print(f"✅ Trained model saved to: {model_path}")

# Save the target encoder (final_result_encoder)
target_encoder_path = models_path / "final_result_encoder.pkl"
joblib.dump(target_encoder, target_encoder_path)
print(f"✅ Target encoder saved to: {target_encoder_path}")

# Save the feature encoders
feature_encoders_path = models_path / "feature_encoders.pkl"
joblib.dump(feature_encoders, feature_encoders_path)
print(f"✅ Feature encoders saved to: {feature_encoders_path}")

# Save feature column names for reference
feature_columns_path = models_path / "feature_columns.pkl"
joblib.dump(feature_columns, feature_columns_path)
print(f"✅ Feature columns saved to: {feature_columns_path}")

print("\n🎉 All models and encoders saved successfully!")


=== Saving Models and Encoders ===
✅ Trained model saved to: ..\6_Models\trained_model.pkl
✅ Target encoder saved to: ..\6_Models\final_result_encoder.pkl
✅ Feature encoders saved to: ..\6_Models\feature_encoders.pkl
✅ Feature columns saved to: ..\6_Models\feature_columns.pkl

🎉 All models and encoders saved successfully!


## 7. Summary

This notebook successfully implemented Milestone 1 of the AI Academic Mentor project:

### What we accomplished:
1. **Data Loading**: Loaded studentInfo.csv, studentAssessment.csv, and assessments.csv
2. **Feature Engineering**: Calculated avg_score and num_assessments per student
3. **Data Merging**: Combined aggregated features with student information
4. **Data Cleaning**: Handled missing values appropriately
5. **Feature Selection**: Selected key features for modeling
6. **Label Encoding**: Encoded categorical features and target variable
7. **Model Training**: Trained GradientBoostingClassifier with good accuracy
8. **Model Persistence**: Saved all models and encoders for deployment

### Next Steps:
- Port this logic to Python scripts in 4_Src_Code/
- Create helper functions for single student prediction
- Integrate with the agentic AI pipeline
